## 0. Mục đích và cách sử dụng

Notebook này chỉ kiểm định release cho pipeline LegalQA sau Giai đoạn 1–5. Nó không sinh full submission 1.000 câu, không dựng lại Dense, không tự hạ xuống BM25-only và không sửa notebook production.

Chạy tuần tự từ trên xuống trên Kaggle GPU T4. Các stage được checkpoint theo commit và config fingerprint; mọi trạng thái FAIL/PENDING/SKIPPED đều làm release_ready = false.

In [ ]:
print("LEGALQA RELEASE GATE — TEST ONLY")
print("Không chạy full public generation; không rebuild Dense; không fallback BM25-only.")

## 1. Cấu hình

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import re
import shutil
import sqlite3
import statistics
import subprocess
import sys
import time
import zipfile
from collections import Counter
from pathlib import Path
from typing import Any

from IPython.display import FileLink, Markdown, display

FORCE_REBUILD_INDEX = False
BUILD_DENSE_INDEX = False
ALLOW_RETRIEVAL_FALLBACK = False
STOP_ON_FAIL = True

RUN_UNIT_TESTS = True
RUN_PUBLIC_RETRIEVAL = True
RUN_SMOKE30 = True
RUN_VALIDATION_100 = True
RUN_VALIDATION_300 = True

MODE = "hybrid_rag"
KNN_THRESHOLD = 0.72
GUARDED_KNN_THRESHOLD = 0.90

BM25_TOP_K = 50
DENSE_TOP_K = 50
RRF_K = 60
RRF_TOP_K = 50
RERANKER_CANDIDATE_K = 20
RERANK_TOP_K = 3

DENSE_QUERY_MAX_LENGTH = 256
RERANKER_MAX_LENGTH = 1024
MAX_INPUT_TOKENS = 7168
MAX_NEW_TOKENS = 512
TOKEN_LIMIT_RETRY_TOKENS = 768
MAX_LONG_ANSWER_WORDS = 640

SEED = 2026

RETRIEVAL_MEDIAN_MAX_SECONDS = 2.0
SMOKE_MEDIAN_MAX_SECONDS = 15.5
TOKEN_LIMIT_RATE_MAX = 0.05
REFUSAL_RATE_MAX = 0.02
EXTRACTIVE_FALLBACK_RATE_MAX = 0.10
OVER_800_WORD_RATE_MAX = 0.05
P90_WORDS_MAX = 700

MANUAL_REVIEW_APPROVED_IDS = []
BASELINE_VALIDATION_300_PATH = None  # Điền path tuyệt đối trong Kaggle Input khi có baseline tương thích.

REPO_URL = "https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git"
EMBEDDING_MODEL_ID = "AITeamVN/Vietnamese_Embedding_v2"
RERANKER_MODEL_ID = "AITeamVN/Vietnamese_Reranker"
GENERATOR_MODEL_ID = "AITeamVN/Vi-Qwen2-1.5B-RAG"
DEVICE = "auto"
MODEL_MARKER = ".legalqa_model.json"

REGRESSION_IDS = [
    "80189", "34235", "31969", "123257", "35853", "154891",
    "117399", "108017", "18645", "129215", "129859",
]
HISTORICAL_RISKY_IDS = [
    "97935", "83665", "63093", "76135",
    "94131", "120127", "103999", "59823",
]
EXPECTED_TARGETS = {
    "34235": {"document_id": "102434", "chunk_no": 306, "required_phrases": ["thời hiệu khiếu nại", "15 ngày"]},
    "62147": {"document_id": "41395", "chunk_no": 3, "required_phrases": ["mức lương cơ sở", "1.800.000 đồng"]},
    "86293": {"document_id": "260328", "chunk_no": 1, "required_phrases": ["quy hoạch điện viii"]},
    "80189": {"document_id": "230689", "chunk_no": 19, "required_phrases": []},
    "135669": {"document_id": "289349", "chunk_no": 18, "required_phrases": []},
}

if FORCE_REBUILD_INDEX or BUILD_DENSE_INDEX or ALLOW_RETRIEVAL_FALLBACK:
    raise RuntimeError("Release gate chỉ chấp nhận cache có sẵn và không cho retrieval fallback.")
if MODE != "hybrid_rag" or GUARDED_KNN_THRESHOLD != 0.90:
    raise RuntimeError("Cấu hình release bắt buộc mode=hybrid_rag và guarded threshold=0.90.")

KAGGLE = Path("/kaggle/working").is_dir()
INPUT_ROOT = Path("/kaggle/input") if KAGGLE else Path(".").resolve()
ROOT_WORK = Path("/kaggle/working/legalqa-test") if KAGGLE else Path("artifacts/legalqa-test").resolve()
REPO_DIR = Path("/kaggle/working/uit-dsc-2026-task2-legalqa") if KAGGLE else Path(".").resolve()
ROOT_WORK.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def json_hash(value: Any) -> str:
    raw = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def read_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))

def atomic_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

def run_stream(command: list[str], *, cwd: Path | None = None, log_path: Path | None = None) -> int:
    print("$", " ".join(map(str, command)), flush=True)
    lines: list[str] = []
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    if process.stdout is not None:
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
            lines.append(line)
        process.stdout.close()
    return_code = process.wait()
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text("".join(lines), encoding="utf-8")
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code

def nearest_percentile(values: list[int], percentile: float) -> int:
    ordered = sorted(values)
    if not ordered:
        return 0
    return ordered[round(percentile * (len(ordered) - 1))]

GATES: dict[str, dict[str, Any]] = {}
def set_gate(name: str, status: str, actual: Any, threshold: Any, detail: str) -> None:
    if status not in {"PASS", "FAIL", "PENDING", "SKIPPED"}:
        raise ValueError(f"Gate status không hợp lệ: {status}")
    GATES[name] = {
        "status": status,
        "actual": actual,
        "threshold": threshold,
        "detail": detail,
    }

print(f"Input root: {INPUT_ROOT}")
print(f"Working root: {ROOT_WORK}")

## 2. Clone repo và xác nhận commit

In [ ]:
if KAGGLE:
    if (REPO_DIR / ".git").is_dir():
        run_stream(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO_DIR)
    elif REPO_DIR.exists():
        raise RuntimeError(f"{REPO_DIR} đã tồn tại nhưng không phải Git repo; không tự xóa.")
    else:
        run_stream(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)])

COMMIT_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True, encoding="utf-8"
).strip()
BRANCH = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True, encoding="utf-8"
).strip()
if KAGGLE and BRANCH != "main":
    raise RuntimeError(f"Phải chạy nhánh main, hiện tại là {BRANCH!r}.")
print(f"Commit SHA: {COMMIT_SHA}")
print(f"Branch: {BRANCH}")

## 3. Cài dependency và kiểm tra GPU

In [ ]:
requirements = REPO_DIR / "requirements-generator.txt"
metrics_requirements = REPO_DIR / "requirements-metrics.txt"
if KAGGLE:
    if not requirements.is_file():
        raise FileNotFoundError(f"Thiếu dependency manifest: {requirements}")
    run_stream([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)], cwd=REPO_DIR)
    if metrics_requirements.is_file():
        run_stream([sys.executable, "-m", "pip", "install", "-q", "-r", str(metrics_requirements)], cwd=REPO_DIR)
    run_stream([sys.executable, "-m", "nltk.downloader", "-q", "wordnet", "omw-1.4"])

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA không khả dụng. Hãy bật GPU T4 trong Kaggle Accelerator.")
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
if KAGGLE and not any("T4" in name.upper() for name in gpu_names):
    raise RuntimeError(f"Notebook được khóa cho Kaggle T4; GPU hiện tại: {gpu_names}")
print("GPU:", gpu_names)

help_process = subprocess.run(
    [sys.executable, "-m", "legalqa_baseline", "predict", "--help"],
    cwd=REPO_DIR,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
if help_process.returncode != 0:
    raise RuntimeError(help_process.stdout + help_process.stderr)
help_text = help_process.stdout + help_process.stderr
required_cli_flags = [
    "--guarded-knn-threshold",
    "--token-limit-retry-tokens",
    "--max-long-answer-words",
    "--reranker-candidate-k",
    "--rerank-top-k",
]
missing_cli_flags = [flag for flag in required_cli_flags if flag not in help_text]

pipeline_source = (REPO_DIR / "legalqa_baseline/pipeline.py").read_text(encoding="utf-8")
text_source = (REPO_DIR / "legalqa_baseline/text.py").read_text(encoding="utf-8")
capability_checks = {
    "knn_exact": '"knn_exact"' in pipeline_source,
    "knn_guarded": '"knn_guarded"' in pipeline_source,
    "generated_retry_768": "generated_retry_" in pipeline_source and TOKEN_LIMIT_RETRY_TOKENS == 768,
    "refusal_recovery": "generated_refusal_recovery" in pipeline_source and "knn_guarded_refusal" in pipeline_source,
    "is_heading_only_answer": "def is_heading_only_answer" in text_source,
    "clean_answer": "def clean_answer" in text_source,
    "audit_has_markdown": '"has_markdown"' in pipeline_source,
    "audit_has_document_slug": '"has_document_slug"' in pipeline_source,
    "audit_has_fake_document_number": '"has_fake_document_number"' in pipeline_source,
}
missing_capabilities = [name for name, available in capability_checks.items() if not available]
if missing_cli_flags or missing_capabilities:
    raise RuntimeError(
        "Kaggle đang clone commit cũ hoặc Giai đoạn 5 chưa được push. "
        f"Thiếu CLI={missing_cli_flags}; thiếu chức năng={missing_capabilities}"
    )
print("CLI/capability contract:", json.dumps(capability_checks, ensure_ascii=False, indent=2))

## 4. Tìm dataset, model và index cache

In [ ]:
def artifact_rank(path: Path) -> tuple[int, int, str]:
    has_manifest = any((parent / "legalqa_artifacts.json").is_file() for parent in path.parents)
    return (0 if has_manifest else 1, len(path.parts), str(path))

INPUT_FILES = [path for path in INPUT_ROOT.rglob("*") if path.is_file()] if INPUT_ROOT.exists() else []
print(f"Đã quét {len(INPUT_FILES):,} file.")

def find_named(names: set[str], *, input_only: bool = True) -> Path | None:
    accepted = {name.casefold() for name in names}
    matches = sorted(
        (path for path in INPUT_FILES if path.name.casefold() in accepted),
        key=artifact_rank,
    )
    if matches:
        return matches[0]
    if not input_only:
        for name in names:
            for folder in (REPO_DIR / "data", REPO_DIR / "artifacts", Path("artifacts").resolve()):
                candidate = folder / name
                if candidate.is_file():
                    return candidate
    return None

def require_named(label: str, names: set[str], *, input_only: bool = True) -> Path:
    path = find_named(names, input_only=input_only)
    if path is None:
        similar = [str(p) for p in INPUT_FILES if any(ext in p.name.lower() for ext in (".sqlite", ".db", ".faiss", ".npy", ".meta.json"))]
        top_dirs = [str(d.name) for d in INPUT_ROOT.iterdir() if d.is_dir()] if INPUT_ROOT.exists() else []
        msg = (
            f"Thiếu {label}: {sorted(names)}.\n"
            f"- INPUT_ROOT: {INPUT_ROOT} (quét được {len(INPUT_FILES)} file)\n"
            f"- Các dataset hiện có trong input: {top_dirs}\n"
            f"- Các file database/vector tìm thấy: {similar[:10]}\n"
            "- HƯỚNG DẪN: Hãy kiểm tra xem bạn đã bấm '+ Add Input' để thêm Kaggle Dataset "
            "chứa index (legalqa.sqlite, legalqa_dense.meta.json,...) vào notebook chưa."
        )
        raise FileNotFoundError(msg)
    return path

def complete_model(folder: Path) -> bool:
    return (folder / "config.json").is_file() and (
        any(folder.glob("*.safetensors")) or any(folder.glob("pytorch_model*.bin"))
    )

def discover_model(repo_id: str) -> dict[str, Any]:
    marked: list[dict[str, Any]] = []
    for marker in (path for path in INPUT_FILES if path.name == MODEL_MARKER):
        try:
            payload = read_json(marker)
        except Exception:
            continue
        if payload.get("repo_id") == repo_id and complete_model(marker.parent):
            marked.append({
                "repo_id": repo_id,
                "path": marker.parent,
                "revision": payload.get("revision"),
                "source": "marker",
            })
    if marked:
        return sorted(marked, key=lambda item: artifact_rank(item["path"]))[0]

    cache_key = ("models--" + repo_id.replace("/", "--")).casefold()
    cached: list[dict[str, Any]] = []
    for config_path in (path for path in INPUT_FILES if path.name == "config.json"):
        folder = config_path.parent
        folded_parts = [part.casefold() for part in folder.parts]
        if cache_key in folded_parts and complete_model(folder):
            revision = folder.name if folder.parent.name == "snapshots" else None
            cached.append({
                "repo_id": repo_id,
                "path": folder,
                "revision": revision,
                "source": "huggingface_snapshot",
            })
    if cached:
        return sorted(cached, key=lambda item: artifact_rank(item["path"]))[0]
    raise FileNotFoundError(f"Thiếu local model snapshot trong Kaggle Input: {repo_id}")

PUBLIC_PATH = require_named(
    "public-official.json",
    {"public-official.json", "public_official.json", "public_test.json"},
    input_only=False,
)
TRAIN_PATH = require_named("train.json", {"train.json"}, input_only=False)
BM25_SOURCE_PATH = require_named(
    "legalqa.sqlite",
    {"legalqa.sqlite", "legalqa.db", "legalqa.sqlite3", "bm25.sqlite", "legalqa_bm25.sqlite"},
    input_only=False,
)
DENSE_META_PATH = require_named(
    "legalqa_dense.meta.json",
    {"legalqa_dense.meta.json", "dense.meta.json"},
    input_only=False,
)
DENSE_INDEX_PATH = Path(str(DENSE_META_PATH).removesuffix(".meta.json"))
DENSE_VECTOR_PATH = next(
    (candidate for candidate in (DENSE_INDEX_PATH.with_suffix(".faiss"), DENSE_INDEX_PATH.with_suffix(".npy")) if candidate.is_file()),
    None,
)
if DENSE_VECTOR_PATH is None:
    raise FileNotFoundError(f"Thiếu Dense vector cạnh {DENSE_META_PATH}")

MODEL_INFO = {
    "embedding": discover_model(EMBEDDING_MODEL_ID),
    "reranker": discover_model(RERANKER_MODEL_ID),
    "generator": discover_model(GENERATOR_MODEL_ID),
}
missing_model_revisions = [key for key, value in MODEL_INFO.items() if not value.get("revision")]
if missing_model_revisions:
    raise RuntimeError(f"Model cache phải ghi revision bất biến; còn thiếu: {missing_model_revisions}")

# Dense loader kiểm tra identity trong manifest bằng repo_id@revision. Dựng cache chỉ bằng
# symlink tới Kaggle Input để vừa chạy offline vừa không đổi identity thành local path.
HF_HUB_CACHE = ROOT_WORK / "hf-hub-release-gate"
HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
def link_hf_snapshot(info: dict[str, Any]) -> None:
    model_root = HF_HUB_CACHE / ("models--" + str(info["repo_id"]).replace("/", "--"))
    revision = str(info["revision"])
    snapshot_link = model_root / "snapshots" / revision
    snapshot_link.parent.mkdir(parents=True, exist_ok=True)
    if snapshot_link.exists() or snapshot_link.is_symlink():
        if snapshot_link.resolve() != Path(info["path"]).resolve():
            raise RuntimeError(f"HF cache link khác snapshot đã khóa: {snapshot_link}")
    else:
        snapshot_link.symlink_to(Path(info["path"]).resolve(), target_is_directory=True)

for model_info in MODEL_INFO.values():
    link_hf_snapshot(model_info)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

EMBEDDING_MODEL = EMBEDDING_MODEL_ID
EMBEDDING_REVISION = str(MODEL_INFO["embedding"]["revision"])
RERANKER_MODEL = str(MODEL_INFO["reranker"]["path"])
GENERATOR_MODEL = str(MODEL_INFO["generator"]["path"])

connection = sqlite3.connect(f"file:{BM25_SOURCE_PATH.resolve()}?mode=ro", uri=True)
try:
    BM25_METADATA = dict(connection.execute("SELECT key, value FROM metadata"))
finally:
    connection.close()
DENSE_PAYLOAD = read_json(DENSE_META_PATH)
DENSE_MANIFEST = DENSE_PAYLOAD.get("manifest", {}) if isinstance(DENSE_PAYLOAD, dict) else {}

BASELINE_SPLIT_PATH = REPO_DIR / "artifacts/baseline_splits_v1.json"
if not BASELINE_SPLIT_PATH.is_file():
    raise FileNotFoundError(f"Thiếu split manifest: {BASELINE_SPLIT_PATH}")
BASELINE_PATH = Path(BASELINE_VALIDATION_300_PATH) if BASELINE_VALIDATION_300_PATH else None

DATASET_HASHES = {
    "public": sha256_file(PUBLIC_PATH),
    "train": sha256_file(TRAIN_PATH),
    "split_manifest": sha256_file(BASELINE_SPLIT_PATH),
}
LOCKED_CONFIG = {
    "commit_sha": COMMIT_SHA,
    "dataset_hashes": DATASET_HASHES,
    "bm25_metadata": BM25_METADATA,
    "dense_manifest": DENSE_MANIFEST,
    "models": {
        key: {
            "repo_id": value["repo_id"],
            "revision": value.get("revision"),
            "path": str(value["path"]),
        }
        for key, value in MODEL_INFO.items()
    },
    "mode": MODE,
    "top_k": {
        "bm25": BM25_TOP_K,
        "dense": DENSE_TOP_K,
        "rrf_k": RRF_K,
        "rrf_top": RRF_TOP_K,
        "reranker_candidate": RERANKER_CANDIDATE_K,
        "reranker_top": RERANK_TOP_K,
    },
    "max_lengths": {
        "dense_query": DENSE_QUERY_MAX_LENGTH,
        "reranker": RERANKER_MAX_LENGTH,
        "generator_input": MAX_INPUT_TOKENS,
        "generator_output": MAX_NEW_TOKENS,
        "token_retry": TOKEN_LIMIT_RETRY_TOKENS,
        "long_answer_words": MAX_LONG_ANSWER_WORDS,
    },
    "guarded_knn_threshold": GUARDED_KNN_THRESHOLD,
    "generation_seed": SEED,
}
CONFIG_FINGERPRINT = json_hash(LOCKED_CONFIG)
RUN_DIR = ROOT_WORK / f"run-{COMMIT_SHA[:12]}-{CONFIG_FINGERPRINT[:12]}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
STATE_PATH = RUN_DIR / "stage_state.json"
STATE = read_json(STATE_PATH) if STATE_PATH.is_file() else {
    "commit_sha": COMMIT_SHA,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "stages": {},
}
if (
    STATE.get("commit_sha") != COMMIT_SHA
    or STATE.get("config_fingerprint") != CONFIG_FINGERPRINT
):
    raise RuntimeError("Run directory chứa state khác fingerprint; không được trộn artifact.")

def artifact_matches_fingerprint(path: Path) -> bool:
    sidecar = Path(str(path) + ".fingerprint.json")
    if not path.is_file() or not sidecar.is_file():
        return False
    try:
        metadata = read_json(sidecar)
        return (
            metadata.get("commit_sha") == COMMIT_SHA
            and metadata.get("config_fingerprint") == CONFIG_FINGERPRINT
            and metadata.get("artifact_sha256") == sha256_file(path)
        )
    except (OSError, ValueError, TypeError, json.JSONDecodeError):
        return False

def can_reuse(stage: str, required_paths: list[Path]) -> bool:
    record = STATE.get("stages", {}).get(stage, {})
    return (
        record.get("status") == "PASS"
        and record.get("commit_sha") == COMMIT_SHA
        and record.get("config_fingerprint") == CONFIG_FINGERPRINT
        and all(artifact_matches_fingerprint(path) for path in required_paths)
    )

def record_stage(stage: str, status: str, details: Any) -> None:
    STATE.setdefault("stages", {})[stage] = {
        "status": status,
        "commit_sha": COMMIT_SHA,
        "config_fingerprint": CONFIG_FINGERPRINT,
        "updated_at_epoch": time.time(),
        "details": details,
    }
    atomic_json(STATE_PATH, STATE)

def tag_artifact(path: Path) -> None:
    if not path.is_file():
        raise FileNotFoundError(path)
    atomic_json(
        Path(str(path) + ".fingerprint.json"),
        {
            "artifact": path.name,
            "artifact_sha256": sha256_file(path),
            "commit_sha": COMMIT_SHA,
            "config_fingerprint": CONFIG_FINGERPRINT,
            "dataset_hashes": DATASET_HASHES,
            "locked_config": LOCKED_CONFIG,
        },
    )

CONFIG_PATH = RUN_DIR / "release_config.json"
atomic_json(CONFIG_PATH, {
    "commit_sha": COMMIT_SHA,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "locked_config": LOCKED_CONFIG,
})
tag_artifact(CONFIG_PATH)
print(f"Run directory: {RUN_DIR}")
print(f"Config fingerprint: {CONFIG_FINGERPRINT}")
print("Models:", json.dumps({k: {**v, "path": str(v["path"])} for k, v in MODEL_INFO.items()}, ensure_ascii=False, indent=2))

## 5. Unit test

In [ ]:
UNIT_LOG = ROOT_WORK / "unit_tests.log"
UNIT_META = Path(str(UNIT_LOG) + ".fingerprint.json")
unit_reusable = False
if UNIT_LOG.is_file() and UNIT_META.is_file():
    previous_meta = read_json(UNIT_META)
    unit_reusable = (
        previous_meta.get("commit_sha") == COMMIT_SHA
        and previous_meta.get("config_fingerprint") == CONFIG_FINGERPRINT
        and previous_meta.get("artifact_sha256") == sha256_file(UNIT_LOG)
    )

if not RUN_UNIT_TESTS:
    set_gate("unit_test_gate", "SKIPPED", "disabled", "PASS", "RUN_UNIT_TESTS=False")
    record_stage("unit_tests", "SKIPPED", GATES["unit_test_gate"])
else:
    if not unit_reusable:
        if UNIT_LOG.is_file():
            preserved = ROOT_WORK / f"unit_tests-{int(time.time())}.log"
            UNIT_LOG.replace(preserved)
            print(f"Đã bảo toàn log cũ: {preserved}")
        command = [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"]
        process = subprocess.run(
            command,
            cwd=REPO_DIR,
            capture_output=True,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        combined = process.stdout + process.stderr
        UNIT_LOG.write_text(combined, encoding="utf-8")
        print(combined)
        atomic_json(UNIT_META, {
            "commit_sha": COMMIT_SHA,
            "config_fingerprint": CONFIG_FINGERPRINT,
            "artifact_sha256": sha256_file(UNIT_LOG),
        })
        return_code = process.returncode
    else:
        combined = UNIT_LOG.read_text(encoding="utf-8")
        print("Reuse unit test log cùng fingerprint.")
        return_code = 0

    match = re.search(r"Ran\s+(\d+)\s+tests?", combined)
    test_count = int(match.group(1)) if match else 0
    unit_pass = (
        return_code == 0
        and "FAILED" not in combined
        and not re.search(r"^ERROR:", combined, flags=re.MULTILINE)
        and test_count >= 174
    )
    set_gate(
        "unit_test_gate",
        "PASS" if unit_pass else "FAIL",
        {"return_code": return_code, "tests": test_count},
        {"return_code": 0, "minimum_tests": 174},
        str(UNIT_LOG),
    )
    record_stage("unit_tests", GATES["unit_test_gate"]["status"], GATES["unit_test_gate"])
    if not unit_pass and STOP_ON_FAIL:
        raise RuntimeError(f"Unit test gate FAIL. Xem {UNIT_LOG}")

## 6. Kiểm tra tính hợp lệ của index

In [ ]:
def expected_model_identity(info: dict[str, Any]) -> str:
    revision = info.get("revision")
    return f"{info['repo_id']}@{revision}" if revision else str(info["repo_id"])

expected_embedding_identity = expected_model_identity(MODEL_INFO["embedding"])
cache_checks = {
    "bm25_schema": BM25_METADATA.get("schema_version") == "5",
    "bm25_corpus_hash_version": BM25_METADATA.get("corpus_hash_version") == "2",
    "bm25_chunks": int(BM25_METADATA.get("chunks", 0)) > 0,
    "bm25_train_samples": int(BM25_METADATA.get("train_samples", 0)) > 0,
    "dense_schema": int(DENSE_MANIFEST.get("schema_version", 0)) >= 5,
    "dense_corpus_hash_version": DENSE_MANIFEST.get("corpus_hash_version") == "2",
    "dense_pooling": DENSE_MANIFEST.get("pooling") == "cls",
    "dense_normalization": DENSE_MANIFEST.get("normalization") == "l2",
    "dense_similarity": DENSE_MANIFEST.get("similarity") == "dot_product",
    "dense_embedding_identity": DENSE_MANIFEST.get("embedding_model") == expected_embedding_identity,
    "corpus_hash_match": DENSE_MANIFEST.get("corpus_sha256") == BM25_METADATA.get("corpus_sha256"),
    "dense_vector_exists": DENSE_VECTOR_PATH.is_file(),
}
cache_pass = all(cache_checks.values())
set_gate(
    "cache_gate",
    "PASS" if cache_pass else "FAIL",
    cache_checks,
    "Tất cả cache contract đều true",
    f"BM25={BM25_SOURCE_PATH}; Dense={DENSE_INDEX_PATH}",
)
record_stage("cache_gate", GATES["cache_gate"]["status"], GATES["cache_gate"])
if not cache_pass:
    raise RuntimeError(
        "Index cache không hợp lệ; release notebook không tạo vector mới. "
        + json.dumps(cache_checks, ensure_ascii=False)
    )

LOCAL_DB_PATH = ROOT_WORK / "legalqa.sqlite"
def sqlite_metadata(path: Path) -> dict[str, str]:
    connection = sqlite3.connect(f"file:{path.resolve()}?mode=ro", uri=True)
    try:
        return dict(connection.execute("SELECT key, value FROM metadata"))
    finally:
        connection.close()

local_valid = (
    LOCAL_DB_PATH.is_file()
    and LOCAL_DB_PATH.stat().st_size == BM25_SOURCE_PATH.stat().st_size
)
if local_valid:
    try:
        local_valid = sqlite_metadata(LOCAL_DB_PATH) == BM25_METADATA
    except sqlite3.Error:
        local_valid = False

if not local_valid:
    if LOCAL_DB_PATH.exists():
        preserved = ROOT_WORK / f"legalqa-{int(time.time())}.sqlite"
        LOCAL_DB_PATH.replace(preserved)
        print(f"Đã bảo toàn SQLite local cũ: {preserved}")
    copying = ROOT_WORK / f"legalqa-{CONFIG_FINGERPRINT[:12]}.copying"
    shutil.copyfile(BM25_SOURCE_PATH, copying)
    if copying.stat().st_size != BM25_SOURCE_PATH.stat().st_size:
        raise IOError("Bản sao SQLite không đủ kích thước")
    copying.replace(LOCAL_DB_PATH)
if sqlite_metadata(LOCAL_DB_PATH) != BM25_METADATA:
    raise RuntimeError("SQLite local không khớp metadata nguồn")
DB_PATH = LOCAL_DB_PATH
print(json.dumps(cache_checks, ensure_ascii=False, indent=2))
print(f"BM25 local: {DB_PATH}")

## 7. Retrieval-only trên 1.000 public questions

In [ ]:
from legalqa_baseline.text import (
    clean_answer,
    is_heading_only_answer,
    is_long_form_question,
    is_refusal_answer,
    output_artifact_flags,
    possibly_cut,
)

PUBLIC_DATA = read_json(PUBLIC_PATH)
if not isinstance(PUBLIC_DATA, dict) or len(PUBLIC_DATA) != 1000:
    raise RuntimeError(f"Public phải có đúng 1.000 câu, thực tế={len(PUBLIC_DATA) if isinstance(PUBLIC_DATA, dict) else 'invalid'}")
if not all(
    isinstance(item, dict) and isinstance(item.get("question"), str) and item["question"].strip()
    for item in PUBLIC_DATA.values()
):
    raise ValueError("public-official.json sai schema")

def retrieval_args(input_path: Path) -> list[str]:
    return [
        "--input", str(input_path),
        "--db", str(DB_PATH),
        "--knn-threshold", str(KNN_THRESHOLD),
        "--guarded-knn-threshold", str(GUARDED_KNN_THRESHOLD),
        "--context-top-k", str(RERANK_TOP_K),
        "--bm25-top-k", str(BM25_TOP_K),
        "--dense-top-k", str(DENSE_TOP_K),
        "--rrf-k", str(RRF_K),
        "--rrf-top-k", str(RRF_TOP_K),
        "--reranker-candidate-k", str(RERANKER_CANDIDATE_K),
        "--rerank-top-k", str(RERANK_TOP_K),
        "--dense-query-max-length", str(DENSE_QUERY_MAX_LENGTH),
        "--reranker-max-length", str(RERANKER_MAX_LENGTH),
        "--max-long-answer-words", str(MAX_LONG_ANSWER_WORDS),
        "--dense-index", str(DENSE_INDEX_PATH),
        "--embedding-model", EMBEDDING_MODEL,
        "--embedding-revision", EMBEDDING_REVISION,
        "--reranker-model", RERANKER_MODEL,
        "--device", DEVICE,
    ]

PUBLIC_RETRIEVAL_PATH = RUN_DIR / "retrieval_public_1000.json"
PUBLIC_RETRIEVAL_LOG = RUN_DIR / "retrieval_public_1000.log"
if not RUN_PUBLIC_RETRIEVAL:
    PUBLIC_RETRIEVAL = None
    set_gate("public_retrieval_gate", "SKIPPED", "disabled", "PASS", "RUN_PUBLIC_RETRIEVAL=False")
    record_stage("public_retrieval", "SKIPPED", GATES["public_retrieval_gate"])
else:
    if not can_reuse("public_retrieval", [PUBLIC_RETRIEVAL_PATH, PUBLIC_RETRIEVAL_LOG]):
        command = [
            sys.executable, "-m", "legalqa_baseline", "diagnose-retrieval",
            *retrieval_args(PUBLIC_PATH),
            "--output", str(PUBLIC_RETRIEVAL_PATH),
        ]
        run_stream(command, cwd=REPO_DIR, log_path=PUBLIC_RETRIEVAL_LOG)
    PUBLIC_RETRIEVAL = read_json(PUBLIC_RETRIEVAL_PATH)
    items = PUBLIC_RETRIEVAL.get("items", [])
    expected_sizes = {"top50": RRF_TOP_K, "top20": RERANKER_CANDIDATE_K, "top3": RERANK_TOP_K}
    stage_shortfalls: list[dict[str, Any]] = []
    bad_status_ids: list[str] = []
    dense_inactive_ids: list[str] = []
    reranker_inactive_ids: list[str] = []
    for item in items:
        qid = str(item.get("id"))
        diagnostics = item.get("diagnostic_candidates", {})
        for stage, minimum in expected_sizes.items():
            count = len(diagnostics.get(stage, [])) if isinstance(diagnostics, dict) else 0
            if count < minimum:
                stage_shortfalls.append({"id": qid, "stage": stage, "count": count, "required": minimum})
        if item.get("status") != "ok":
            bad_status_ids.append(qid)
        trace = item.get("retrieval_trace", {})
        if trace.get("dense", {}).get("status") != "ok":
            dense_inactive_ids.append(qid)
        if trace.get("reranker_pool", {}).get("status") != "ok":
            reranker_inactive_ids.append(qid)

    summary = PUBLIC_RETRIEVAL.get("summary", {})
    median_total = float(summary.get("median_stage_seconds", {}).get("total", float("inf")))
    public_pass = (
        len(items) == 1000
        and int(summary.get("samples", 0)) == 1000
        and summary.get("generator_called") is False
        and median_total <= RETRIEVAL_MEDIAN_MAX_SECONDS
        and not stage_shortfalls
        and not bad_status_ids
        and not dense_inactive_ids
        and not reranker_inactive_ids
    )
    public_health = {
        "commit_sha": COMMIT_SHA,
        "config_fingerprint": CONFIG_FINGERPRINT,
        "samples": len(items),
        "generator_called": summary.get("generator_called"),
        "median_total_seconds": median_total,
        "stage_shortfalls": stage_shortfalls,
        "exception_or_empty_ids": bad_status_ids,
        "dense_inactive_ids": dense_inactive_ids,
        "reranker_inactive_ids": reranker_inactive_ids,
        "note": "Public không có reference answer; đây là retrieval health/audit, không phải Recall@K.",
    }
    PUBLIC_RETRIEVAL["_release_gate"] = public_health
    atomic_json(PUBLIC_RETRIEVAL_PATH, PUBLIC_RETRIEVAL)
    tag_artifact(PUBLIC_RETRIEVAL_PATH)
    tag_artifact(PUBLIC_RETRIEVAL_LOG)
    set_gate(
        "public_retrieval_gate",
        "PASS" if public_pass else "FAIL",
        public_health,
        {"samples": 1000, "median_seconds_max": RETRIEVAL_MEDIAN_MAX_SECONDS, "all_stages_active": True},
        str(PUBLIC_RETRIEVAL_PATH),
    )
    record_stage("public_retrieval", GATES["public_retrieval_gate"]["status"], GATES["public_retrieval_gate"])
    print(json.dumps(public_health, ensure_ascii=False, indent=2))

## 8. Tạo smoke set 30 câu cố định

In [ ]:
if not isinstance(PUBLIC_DATA, dict):
    raise RuntimeError("Public data chưa sẵn sàng")

missing_regression = [qid for qid in REGRESSION_IDS if qid not in PUBLIC_DATA]
if missing_regression:
    raise RuntimeError(f"Thiếu regression IDs trong public: {missing_regression}")

selected_ids = list(REGRESSION_IDS)
selection_reason = {qid: "regression" for qid in REGRESSION_IDS}
historical_selected = [
    qid for qid in HISTORICAL_RISKY_IDS
    if qid in PUBLIC_DATA and qid not in selection_reason
][:4]
if len(historical_selected) != 4:
    raise RuntimeError(f"Cần đúng 4 historical risky IDs, chỉ tìm thấy {historical_selected}")
for qid in historical_selected:
    selected_ids.append(qid)
    selection_reason[qid] = "historical_token_limit_or_risky"

normal_pool = [
    str(qid)
    for qid, item in PUBLIC_DATA.items()
    if str(qid) not in selection_reason
    and not is_long_form_question(str(item.get("question") or ""))
]
normal_pool.sort()
rng = random.Random(SEED)
rng.shuffle(normal_pool)
random_ids = normal_pool[:15]
if len(random_ids) != 15:
    raise RuntimeError("Không đủ 15 random normal IDs")
for qid in random_ids:
    selected_ids.append(qid)
    selection_reason[qid] = "random_normal"

if len(selected_ids) != 30 or len(set(selected_ids)) != 30:
    raise RuntimeError(f"Smoke set phải đúng 30 ID duy nhất, thực tế={len(selected_ids)}")

SMOKE_DATA = {qid: PUBLIC_DATA[qid] for qid in selected_ids}
SMOKE_INPUT_PATH = RUN_DIR / "public_smoke30.json"
SMOKE_MANIFEST_PATH = RUN_DIR / "smoke30_manifest.json"
atomic_json(SMOKE_INPUT_PATH, SMOKE_DATA)
SMOKE_MANIFEST = {
    "seed": SEED,
    "commit_sha": COMMIT_SHA,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "dataset_hashes": DATASET_HASHES,
    "ids": [
        {
            "id": qid,
            "reason": selection_reason[qid],
            "question": PUBLIC_DATA[qid]["question"],
        }
        for qid in selected_ids
    ],
}
atomic_json(SMOKE_MANIFEST_PATH, SMOKE_MANIFEST)
tag_artifact(SMOKE_INPUT_PATH)
tag_artifact(SMOKE_MANIFEST_PATH)
record_stage("smoke_selection", "PASS", {
    "count": len(selected_ids),
    "groups": dict(Counter(selection_reason.values())),
    "manifest": str(SMOKE_MANIFEST_PATH),
})
print("Smoke groups:", dict(Counter(selection_reason.values())))
print("Smoke IDs:", selected_ids)

## 9. Retrieval gate cho 5 known problems

In [ ]:
RETRIEVAL_SMOKE_PATH = RUN_DIR / "retrieval_smoke30.json"
RETRIEVAL_GATE_PATH = RUN_DIR / "retrieval_gate.json"

if PUBLIC_RETRIEVAL is None:
    retrieval_known_pass = False
    retrieval_gate_report = {"status": "SKIPPED", "reason": "public retrieval disabled"}
else:
    public_item_by_id = {str(item["id"]): item for item in PUBLIC_RETRIEVAL.get("items", [])}
    missing_target_questions = [qid for qid in EXPECTED_TARGETS if qid not in public_item_by_id]
    if missing_target_questions:
        raise RuntimeError(f"Public retrieval thiếu known targets: {missing_target_questions}")

    from legalqa_baseline.storage import SearchIndex

    def target_rank(qid: str, stage: str, expected: dict[str, Any]) -> int | None:
        candidates = public_item_by_id[qid].get("diagnostic_candidates", {}).get(stage, [])
        for candidate in candidates:
            ids = {
                str(candidate.get("document_id") or ""),
                str(candidate.get("context_id") or ""),
            }
            if str(expected["document_id"]) not in ids:
                continue
            if int(candidate.get("chunk_no", -1)) != int(expected["chunk_no"]):
                continue
            return int(candidate.get("rank", 0))
        return None

    checks: dict[str, Any] = {}
    with SearchIndex(DB_PATH) as index:
        for qid, expected in EXPECTED_TARGETS.items():
            chunks = index.get_context_chunks(
                str(expected["document_id"]),
                chunk_nos=[int(expected["chunk_no"])],
            )
            exact_chunk = next(
                (
                    chunk for chunk in chunks
                    if str(chunk.get("context_id")) == str(expected["document_id"])
                    and int(chunk.get("chunk_no", -1)) == int(expected["chunk_no"])
                ),
                None,
            )
            normalized_text = " ".join(str((exact_chunk or {}).get("text") or "").casefold().split())
            phrase_checks = {
                phrase: phrase.casefold() in normalized_text
                for phrase in expected.get("required_phrases", [])
            }
            ranks = {
                stage: target_rank(qid, stage, expected)
                for stage in ("top50", "top20", "top3")
            }
            item_pass = (
                exact_chunk is not None
                and ranks["top50"] is not None
                and ranks["top20"] is not None
                and ranks["top3"] == 1
                and all(phrase_checks.values())
            )
            checks[qid] = {
                "expected": expected,
                "ranks": ranks,
                "required_phrase_checks": phrase_checks,
                "exact_chunk_found": exact_chunk is not None,
                "pass": item_pass,
            }

    retrieval_known_pass = len(checks) == 5 and all(item["pass"] for item in checks.values())
    retrieval_gate_report = {
        "commit_sha": COMMIT_SHA,
        "config_fingerprint": CONFIG_FINGERPRINT,
        "status": "PASS" if retrieval_known_pass else "FAIL",
        "passed": sum(bool(item["pass"]) for item in checks.values()),
        "required": 5,
        "checks": checks,
    }
    retrieval_smoke_payload = {
        "summary": {
            "samples": 30,
            "commit_sha": COMMIT_SHA,
            "config_fingerprint": CONFIG_FINGERPRINT,
        },
        "items": [public_item_by_id[qid] for qid in selected_ids],
        "known_problem_items": [public_item_by_id[qid] for qid in EXPECTED_TARGETS],
    }
    atomic_json(RETRIEVAL_SMOKE_PATH, retrieval_smoke_payload)
    atomic_json(RETRIEVAL_GATE_PATH, retrieval_gate_report)
    tag_artifact(RETRIEVAL_SMOKE_PATH)
    tag_artifact(RETRIEVAL_GATE_PATH)

set_gate(
    "known_problem_retrieval_gate",
    "PASS" if retrieval_known_pass else ("SKIPPED" if PUBLIC_RETRIEVAL is None else "FAIL"),
    retrieval_gate_report,
    "5/5 targets có Top50, Top20 và final Top1; phrase nằm trong đúng chunk",
    str(RETRIEVAL_GATE_PATH),
)
record_stage(
    "known_problem_retrieval",
    GATES["known_problem_retrieval_gate"]["status"],
    GATES["known_problem_retrieval_gate"],
)
print(json.dumps(retrieval_gate_report, ensure_ascii=False, indent=2))
if not retrieval_known_pass:
    print("Known-problem retrieval FAIL/SKIPPED: validation 300 và release đã bị khóa; report vẫn tiếp tục được xuất.")

## 10. Smoke generation 30 câu

In [ ]:
SMOKE_SUBMISSION_PATH = RUN_DIR / "submission_smoke30.json"
SMOKE_AUDIT_PATH = RUN_DIR / "submission_smoke30.audit.jsonl"
SMOKE_CHECKPOINT_PATH = SMOKE_SUBMISSION_PATH.with_suffix(".checkpoint.json")
SMOKE_LOG_PATH = RUN_DIR / "smoke30_generation.log"
SMOKE_RUN_CONFIG_PATH = RUN_DIR / "smoke30_run_config.json"

if not RUN_SMOKE30:
    set_gate("smoke_execution_gate", "SKIPPED", "disabled", "PASS", "RUN_SMOKE30=False")
    record_stage("smoke_generation", "SKIPPED", GATES["smoke_execution_gate"])
else:
    smoke_run_config = {
        "commit_sha": COMMIT_SHA,
        "config_fingerprint": CONFIG_FINGERPRINT,
        "input_sha256": sha256_file(SMOKE_INPUT_PATH),
    }
    if SMOKE_RUN_CONFIG_PATH.is_file() and read_json(SMOKE_RUN_CONFIG_PATH) != smoke_run_config:
        raise RuntimeError("Smoke checkpoint có config fingerprint khác; không được resume.")
    atomic_json(SMOKE_RUN_CONFIG_PATH, smoke_run_config)

    if not can_reuse("smoke_generation", [SMOKE_SUBMISSION_PATH, SMOKE_AUDIT_PATH, SMOKE_LOG_PATH]):
        command = [
            sys.executable, "-m", "legalqa_baseline", "predict",
            *retrieval_args(SMOKE_INPUT_PATH),
            "--output", str(SMOKE_SUBMISSION_PATH),
            "--audit-output", str(SMOKE_AUDIT_PATH),
            "--mode", MODE,
            "--generator-model", GENERATOR_MODEL,
            "--max-new-tokens", str(MAX_NEW_TOKENS),
            "--max-input-tokens", str(MAX_INPUT_TOKENS),
            "--token-limit-retry-tokens", str(TOKEN_LIMIT_RETRY_TOKENS),
            "--generation-seed", str(SEED),
            "--checkpoint-interval", "1",
        ]
        if SMOKE_CHECKPOINT_PATH.is_file():
            command.append("--resume")
        run_stream(command, cwd=REPO_DIR, log_path=SMOKE_LOG_PATH)

    if not SMOKE_SUBMISSION_PATH.is_file() or not SMOKE_AUDIT_PATH.is_file():
        raise RuntimeError("Smoke command thành công nhưng thiếu submission/audit")
    for artifact in (SMOKE_SUBMISSION_PATH, SMOKE_AUDIT_PATH, SMOKE_LOG_PATH, SMOKE_RUN_CONFIG_PATH):
        tag_artifact(artifact)
    record_stage("smoke_generation", "PASS", {
        "submission": str(SMOKE_SUBMISSION_PATH),
        "audit": str(SMOKE_AUDIT_PATH),
    })
    set_gate("smoke_execution_gate", "PASS", 30, 30, str(SMOKE_SUBMISSION_PATH))
    print(f"Smoke submission: {SMOKE_SUBMISSION_PATH}")
    print(f"Smoke audit: {SMOKE_AUDIT_PATH}")

## 11. Automatic quality gate

In [ ]:
SMOKE_QUALITY_PATH = RUN_DIR / "smoke30_quality_report.json"

if not RUN_SMOKE30:
    SMOKE_QUALITY = {"status": "SKIPPED", "pass": False}
    smoke_automatic_pass = False
    set_gate("smoke_automatic_gate", "SKIPPED", "disabled", "PASS", "RUN_SMOKE30=False")
else:
    predictions = read_json(SMOKE_SUBMISSION_PATH)
    audit_rows = [
        json.loads(line)
        for line in SMOKE_AUDIT_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    audit_by_id = {str(row.get("id")): row for row in audit_rows}
    expected_id_set = set(selected_ids)

    schema_ids = [
        qid for qid, item in predictions.items()
        if not isinstance(item, dict)
        or set(item) != {"answer"}
        or not isinstance(item.get("answer"), str)
        or not item["answer"].strip()
    ]
    id_mismatch = {
        "missing_prediction": sorted(expected_id_set - set(predictions)),
        "extra_prediction": sorted(set(predictions) - expected_id_set),
        "missing_audit": sorted(expected_id_set - set(audit_by_id)),
        "extra_audit": sorted(set(audit_by_id) - expected_id_set),
        "duplicate_audit_rows": len(audit_rows) != len(audit_by_id),
    }
    answers = {
        qid: str(predictions[qid]["answer"]).strip()
        for qid in selected_ids
        if qid in predictions and isinstance(predictions[qid], dict)
    }
    artifact_by_id = {
        qid: sorted(output_artifact_flags(answer))
        for qid, answer in answers.items()
        if output_artifact_flags(answer)
    }
    markdown_ids = [qid for qid, flags in artifact_by_id.items() if "markdown" in flags]
    slug_ids = [qid for qid, flags in artifact_by_id.items() if "document_slug" in flags]
    fake_number_ids = [
        qid for qid, flags in artifact_by_id.items()
        if "fake_document_number_or_page_id" in flags
    ]
    boilerplate_ids = [
        qid for qid, flags in artifact_by_id.items()
        if "answer_boilerplate" in flags
    ]
    url_ids = [qid for qid, answer in answers.items() if re.search(r"https?://", answer, re.I)]
    refusal_ids = [qid for qid, answer in answers.items() if is_refusal_answer(answer)]
    possibly_cut_ids = [qid for qid, answer in answers.items() if possibly_cut(answer)]
    heading_only_ids = [
        qid for qid, answer in answers.items()
        if str(audit_by_id.get(qid, {}).get("route", "")).startswith("extractive")
        and is_heading_only_answer(answer)
    ]

    word_counts = {qid: len(answer.split()) for qid, answer in answers.items()}
    lengths = list(word_counts.values())
    p90_words = nearest_percentile(lengths, 0.90)
    over_800_ids = [qid for qid, count in word_counts.items() if count > 800]
    over_800_rate = len(over_800_ids) / max(1, len(lengths))
    long_extractive_ids = [
        qid for qid, count in word_counts.items()
        if str(audit_by_id.get(qid, {}).get("route", "")).startswith("extractive")
        and count > MAX_LONG_ANSWER_WORDS
    ]

    final_token_limit_ids = [
        qid for qid, row in audit_by_id.items()
        if bool(row.get("hit_token_limit"))
    ]
    initial_token_limit_ids = [
        qid for qid, row in audit_by_id.items()
        if bool(row.get("initial_hit_token_limit"))
    ]
    final_refusal_ids = sorted(set(refusal_ids) | {
        qid for qid, row in audit_by_id.items() if bool(row.get("says_no_information"))
    })
    extractive_fallback_ids = [
        qid for qid, row in audit_by_id.items()
        if row.get("route") == "extractive_fallback"
    ]
    retry_768_ids = [
        qid for qid, row in audit_by_id.items()
        if row.get("route") == "generated_retry_768"
    ]
    recovery_exhausted_invalid_ids = [
        qid for qid, row in audit_by_id.items()
        if row.get("route") == "recovery_exhausted"
        and (
            qid not in answers
            or qid in refusal_ids
            or qid in possibly_cut_ids
            or qid in artifact_by_id
        )
    ]

    generation_timing: dict[str, Any] = {}
    missing_split_timing_ids: list[str] = []
    for qid, row in audit_by_id.items():
        attempts = int(row.get("generation_attempts") or 0)
        combined = float(row.get("stage_seconds", {}).get("generation", 0.0))
        initial_seconds = row.get("initial_generation_seconds")
        retry_seconds = row.get("retry_generation_seconds")
        if attempts <= 1:
            initial_seconds = combined
            retry_seconds = 0.0
        elif initial_seconds is None and retry_seconds is None:
            initial_seconds = combined
            retry_seconds = 0.0
        elif initial_seconds is None or retry_seconds is None:
            missing_split_timing_ids.append(qid)
        generation_timing[qid] = {
            "attempts": attempts,
            "combined_seconds": combined,
            "initial_seconds": initial_seconds,
            "retry_seconds": retry_seconds,
        }

    total_seconds = [
        float(row.get("stage_seconds", {}).get("total", 0.0))
        for row in audit_rows
    ]
    median_total = statistics.median(total_seconds) if total_seconds else float("inf")
    token_limit_rate = len(final_token_limit_ids) / max(1, len(audit_by_id))
    refusal_rate = len(final_refusal_ids) / max(1, len(audit_by_id))
    extractive_fallback_rate = len(extractive_fallback_ids) / max(1, len(audit_by_id))

    conditions = {
        "exactly_30_ids": len(predictions) == 30 and not any(
            value for key, value in id_mismatch.items() if key != "duplicate_audit_rows"
        ) and not id_mismatch["duplicate_audit_rows"],
        "schema_valid": not schema_ids,
        "no_output_artifacts": not artifact_by_id and not url_ids,
        "no_refusal": not final_refusal_ids,
        "no_possibly_cut": not possibly_cut_ids,
        "no_heading_only": not heading_only_ids,
        "extractive_length_bounded": not long_extractive_ids,
        "p90_words": p90_words <= P90_WORDS_MAX,
        "over_800_rate": over_800_rate < OVER_800_WORD_RATE_MAX,
        "median_total_seconds": median_total <= SMOKE_MEDIAN_MAX_SECONDS,
        "token_limit_rate": token_limit_rate < TOKEN_LIMIT_RATE_MAX,
        "refusal_rate": refusal_rate < REFUSAL_RATE_MAX,
        "extractive_fallback_rate": extractive_fallback_rate < EXTRACTIVE_FALLBACK_RATE_MAX,
        "recovery_exhausted_outputs_valid": not recovery_exhausted_invalid_ids,
        "retry_timing_observable": not missing_split_timing_ids,
    }
    smoke_automatic_pass = all(conditions.values())
    SMOKE_QUALITY = {
        "commit_sha": COMMIT_SHA,
        "config_fingerprint": CONFIG_FINGERPRINT,
        "status": "PASS" if smoke_automatic_pass else "FAIL",
        "pass": smoke_automatic_pass,
        "conditions": conditions,
        "routes": dict(Counter(str(row.get("route")) for row in audit_rows)),
        "length": {
            "p50": statistics.median(lengths) if lengths else None,
            "p90": p90_words,
            "max": max(lengths) if lengths else None,
            "over_800_rate": over_800_rate,
            "over_800_ids": over_800_ids,
        },
        "timing": {
            "median_total_seconds": median_total,
            "per_id_generation": generation_timing,
            "missing_initial_retry_split_ids": missing_split_timing_ids,
        },
        "routing": {
            "token_limit_rate": token_limit_rate,
            "refusal_rate": refusal_rate,
            "extractive_fallback_rate": extractive_fallback_rate,
            "initial_token_limit_ids": initial_token_limit_ids,
            "final_token_limit_ids": final_token_limit_ids,
            "generated_retry_768_ids": retry_768_ids,
            "extractive_fallback_ids": extractive_fallback_ids,
        },
        "violations": {
            "schema_ids": schema_ids,
            "id_mismatch": id_mismatch,
            "markdown_ids": markdown_ids,
            "url_ids": url_ids,
            "slug_ids": slug_ids,
            "fake_document_number_ids": fake_number_ids,
            "boilerplate_ids": boilerplate_ids,
            "refusal_ids": final_refusal_ids,
            "possibly_cut_ids": possibly_cut_ids,
            "heading_only_ids": heading_only_ids,
            "long_extractive_ids": long_extractive_ids,
            "recovery_exhausted_invalid_ids": recovery_exhausted_invalid_ids,
        },
    }
    atomic_json(SMOKE_QUALITY_PATH, SMOKE_QUALITY)
    tag_artifact(SMOKE_QUALITY_PATH)
    set_gate(
        "smoke_automatic_gate",
        "PASS" if smoke_automatic_pass else "FAIL",
        SMOKE_QUALITY,
        {
            "median_seconds_max": SMOKE_MEDIAN_MAX_SECONDS,
            "p90_words_max": P90_WORDS_MAX,
            "over_800_rate_max": OVER_800_WORD_RATE_MAX,
            "token_limit_rate_max": TOKEN_LIMIT_RATE_MAX,
            "refusal_rate_max": REFUSAL_RATE_MAX,
            "extractive_fallback_rate_max": EXTRACTIVE_FALLBACK_RATE_MAX,
        },
        str(SMOKE_QUALITY_PATH),
    )
    record_stage("smoke_quality", GATES["smoke_automatic_gate"]["status"], GATES["smoke_automatic_gate"])
    print(json.dumps(SMOKE_QUALITY, ensure_ascii=False, indent=2))

## 12. Review thủ công

In [ ]:
MANUAL_REVIEW_PATH = RUN_DIR / "manual_review.json"
required_manual_ids = list(dict.fromkeys(REGRESSION_IDS + random_ids))
if len(required_manual_ids) != 26:
    raise RuntimeError(f"Manual review phải có 26 ID, thực tế={len(required_manual_ids)}")

approved_ids = {str(qid) for qid in MANUAL_REVIEW_APPROVED_IDS}
missing_approval_ids = sorted(set(required_manual_ids) - approved_ids)
manual_status = "PASS" if not missing_approval_ids else "PENDING"

review_items: list[dict[str, Any]] = []
if RUN_SMOKE30:
    if "predictions" not in globals() and SMOKE_SUBMISSION_PATH.is_file():
        predictions = read_json(SMOKE_SUBMISSION_PATH)
    if "audit_by_id" not in globals() and SMOKE_AUDIT_PATH.is_file():
        audit_rows = [
            json.loads(line)
            for line in SMOKE_AUDIT_PATH.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        audit_by_id = {str(row.get("id")): row for row in audit_rows}
    public_retrieval_by_id = {
        str(item["id"]): item
        for item in (PUBLIC_RETRIEVAL or {}).get("items", [])
    }
    for qid in required_manual_ids:
        answer = predictions[qid]["answer"]
        audit_row = audit_by_id[qid]
        top_candidates = public_retrieval_by_id.get(qid, {}).get("diagnostic_candidates", {}).get("top3", [])
        top = top_candidates[0] if top_candidates else {}
        knn_similarity = None
        if str(audit_row.get("route", "")).startswith("knn"):
            from legalqa_baseline.pipeline import question_similarity
            from legalqa_baseline.storage import SearchIndex
            with SearchIndex(DB_PATH) as index:
                neighbors = index.search_train(PUBLIC_DATA[qid]["question"], top_k=5, exclude_id=qid)
            similarities = [
                question_similarity(PUBLIC_DATA[qid]["question"], str(item.get("question") or ""))
                for item in neighbors
            ]
            knn_similarity = max(similarities) if similarities else None
        item = {
            "id": qid,
            "group": selection_reason[qid],
            "question": PUBLIC_DATA[qid]["question"],
            "answer": answer,
            "route": audit_row.get("route"),
            "answer_words": len(answer.split()),
            "top_document_id": top.get("document_id") or top.get("context_id"),
            "top_chunk_no": top.get("chunk_no"),
            "raw_reranker_score": top.get("rerank_score") or audit_row.get("raw_reranker_score"),
            "final_reranker_score": top.get("final_rerank_score"),
            "knn_similarity": knn_similarity,
            "recovery_strategy": audit_row.get("recovery_strategy"),
            "audit_flags": {
                "output_artifacts": audit_row.get("output_artifacts", []),
                "has_markdown": audit_row.get("has_markdown"),
                "has_document_slug": audit_row.get("has_document_slug"),
                "has_fake_document_number": audit_row.get("has_fake_document_number"),
                "says_no_information": audit_row.get("says_no_information"),
                "possibly_cut": audit_row.get("possibly_cut"),
            },
        }
        review_items.append(item)
        display(Markdown(
            f"### ID {qid} — {item['group']}\n"
            f"- Route: {item['route']}\n"
            f"- Words: {item['answer_words']}\n"
            f"- Top document/chunk: {item['top_document_id']} / {item['top_chunk_no']}\n"
            f"- Raw/final reranker: {item['raw_reranker_score']} / {item['final_reranker_score']}\n"
            f"- KNN similarity: {item['knn_similarity']}\n"
            f"- Recovery: {item['recovery_strategy']}\n"
            f"- Audit flags: {json.dumps(item['audit_flags'], ensure_ascii=False)}\n\n"
            f"**Câu hỏi:** {item['question']}\n\n"
            f"**Câu trả lời nguyên văn:**\n\n{item['answer']}"
        ))
else:
    manual_status = "SKIPPED"

MANUAL_REVIEW_REPORT = {
    "commit_sha": COMMIT_SHA,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "status": manual_status,
    "required_ids": required_manual_ids,
    "approved_ids": sorted(approved_ids & set(required_manual_ids)),
    "missing_approval_ids": missing_approval_ids,
    "items": review_items,
}
atomic_json(MANUAL_REVIEW_PATH, MANUAL_REVIEW_REPORT)
tag_artifact(MANUAL_REVIEW_PATH)
set_gate(
    "manual_review_gate",
    manual_status,
    {"approved": len(required_manual_ids) - len(missing_approval_ids), "required": 26, "missing": missing_approval_ids},
    "MANUAL_REVIEW_APPROVED_IDS chứa đủ 26 ID",
    str(MANUAL_REVIEW_PATH),
)
record_stage("manual_review", manual_status, GATES["manual_review_gate"])
print(f"Manual review gate: {manual_status}; chưa duyệt: {missing_approval_ids}")

## 13. Validation 100

In [ ]:
VALIDATION_100_PATH = RUN_DIR / "validation_100.json"
VALIDATION_100_LOG = RUN_DIR / "validation_100.log"

def validation_command(split_name: str, output_path: Path) -> list[str]:
    limit = int(split_name.rsplit("_", 1)[1])
    return [
        sys.executable, "-m", "legalqa_baseline", "validate",
        "--train", str(TRAIN_PATH),
        "--db", str(DB_PATH),
        "--output", str(output_path),
        "--modes", MODE,
        "--limit", str(limit),
        "--seed", str(SEED),
        "--split-manifest", str(BASELINE_SPLIT_PATH),
        "--split-name", split_name,
        "--regression-input", str(PUBLIC_PATH),
        "--official-metrics",
        *retrieval_args(TRAIN_PATH)[4:],
        "--generator-model", GENERATOR_MODEL,
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--max-input-tokens", str(MAX_INPUT_TOKENS),
        "--token-limit-retry-tokens", str(TOKEN_LIMIT_RETRY_TOKENS),
        "--generation-seed", str(SEED),
    ]

if not RUN_VALIDATION_100:
    VALIDATION_100_REPORT = None
    set_gate("validation_100_execution", "SKIPPED", "disabled", "PASS", "RUN_VALIDATION_100=False")
    record_stage("validation_100_execution", "SKIPPED", GATES["validation_100_execution"])
else:
    if not can_reuse("validation_100_execution", [VALIDATION_100_PATH, VALIDATION_100_LOG]):
        run_stream(
            validation_command("validation_100", VALIDATION_100_PATH),
            cwd=REPO_DIR,
            log_path=VALIDATION_100_LOG,
        )
    VALIDATION_100_REPORT = read_json(VALIDATION_100_PATH)
    VALIDATION_100_REPORT["_release_gate"] = {
        "commit_sha": COMMIT_SHA,
        "config_fingerprint": CONFIG_FINGERPRINT,
    }
    atomic_json(VALIDATION_100_PATH, VALIDATION_100_REPORT)
    tag_artifact(VALIDATION_100_PATH)
    tag_artifact(VALIDATION_100_LOG)
    set_gate("validation_100_execution", "PASS", 100, 100, str(VALIDATION_100_PATH))
    record_stage("validation_100_execution", "PASS", GATES["validation_100_execution"])
    print(f"Validation 100 report: {VALIDATION_100_PATH}")

## 14. Validation 100 gate

In [ ]:
def validation_quality(report: dict[str, Any], split_name: str) -> dict[str, Any]:
    result = report.get("results", {}).get(MODE)
    if not isinstance(result, dict):
        return {"status": "FAIL", "pass": False, "reason": f"Thiếu results.{MODE}"}
    items = result.get("items", [])
    predictions_by_id = {
        str(item.get("id")): str(item.get("prediction") or "")
        for item in items
    }
    empty_ids = [qid for qid, answer in predictions_by_id.items() if not answer.strip()]
    heading_only_ids = [
        str(item.get("id"))
        for item in items
        if str(item.get("route", "")).startswith("extractive")
        and is_heading_only_answer(str(item.get("prediction") or ""))
    ]
    dirty_output_ids = [
        qid for qid, answer in predictions_by_id.items()
        if output_artifact_flags(answer)
    ]
    possibly_cut_ids = [
        qid for qid, answer in predictions_by_id.items()
        if possibly_cut(answer)
    ]
    lengths = [
        int(item.get("length", {}).get("prediction_words") or 0)
        for item in items
    ]
    p50 = statistics.median(lengths) if lengths else 0
    p90 = nearest_percentile(lengths, 0.90)
    maximum = max(lengths) if lengths else 0
    over_800_ids = [
        str(item.get("id"))
        for item, words in zip(items, lengths)
        if words > 800
    ]
    over_800_rate = len(over_800_ids) / max(1, len(items))
    token_limit_ids = [
        str(item.get("id"))
        for item in items
        if bool(item.get("audit", {}).get("hit_token_limit"))
    ]
    refusal_ids = [
        str(item.get("id"))
        for item in items
        if bool(item.get("audit", {}).get("says_no_information"))
        or is_refusal_answer(str(item.get("prediction") or ""))
    ]
    fallback_ids = [
        str(item.get("id"))
        for item in items
        if item.get("route") == "extractive_fallback"
    ]

    dense_inactive_ids: list[str] = []
    reranker_inactive_ids: list[str] = []
    retrieval_fallback_ids: list[str] = []
    for item in items:
        qid = str(item.get("id"))
        trace = item.get("retrieval", {}).get("trace", {})
        dense_status = trace.get("dense", {}).get("status")
        reranker_status = trace.get("reranker_pool", {}).get("status")
        if dense_status != "ok":
            dense_inactive_ids.append(qid)
        if reranker_status != "ok":
            reranker_inactive_ids.append(qid)
        if dense_status not in {"ok"} or reranker_status not in {"ok"}:
            retrieval_fallback_ids.append(qid)

    token_rate = len(token_limit_ids) / max(1, len(items))
    refusal_rate = len(refusal_ids) / max(1, len(items))
    fallback_rate = len(fallback_ids) / max(1, len(items))
    conditions = {
        "sample_count": len(items) == int(split_name.rsplit("_", 1)[1]),
        "no_empty": not empty_ids,
        "no_heading_only": not heading_only_ids,
        "no_artifacts": not dirty_output_ids,
        "no_possibly_cut": not possibly_cut_ids,
        "p90_words": p90 <= P90_WORDS_MAX,
        "over_800_rate": over_800_rate < OVER_800_WORD_RATE_MAX,
        "token_limit_rate": token_rate < TOKEN_LIMIT_RATE_MAX,
        "refusal_rate": refusal_rate < REFUSAL_RATE_MAX,
        "extractive_fallback_rate": fallback_rate < EXTRACTIVE_FALLBACK_RATE_MAX,
        "no_retrieval_fallback": not retrieval_fallback_ids,
        "dense_active": not dense_inactive_ids,
        "reranker_active": not reranker_inactive_ids,
    }
    passed = all(conditions.values())
    return {
        "status": "PASS" if passed else "FAIL",
        "pass": passed,
        "split_name": split_name,
        "samples": len(items),
        "conditions": conditions,
        "metrics": {
            key: result.get(key)
            for key in (
                "competition_meteor", "competition_rougeL",
                "meteor_exact_approx", "rougeL", "answer_token_f1",
                "bleu_4", "exact_match",
            )
        },
        "routes": result.get("routes", {}),
        "retrieval_metrics": result.get("retrieval", {}),
        "routing": {
            "token_limit_rate": token_rate,
            "refusal_rate": refusal_rate,
            "extractive_fallback_rate": fallback_rate,
        },
        "length": {
            "p50": p50,
            "p90": p90,
            "max": maximum,
            "over_800_rate": over_800_rate,
        },
        "violations": {
            "empty_ids": empty_ids,
            "heading_only_ids": heading_only_ids,
            "dirty_output_ids": dirty_output_ids,
            "possibly_cut_ids": possibly_cut_ids,
            "over_800_ids": over_800_ids,
            "token_limit_ids": token_limit_ids,
            "refusal_ids": refusal_ids,
            "extractive_fallback_ids": fallback_ids,
            "retrieval_fallback_ids": retrieval_fallback_ids,
            "dense_inactive_ids": dense_inactive_ids,
            "reranker_inactive_ids": reranker_inactive_ids,
        },
    }

VALIDATION_100_GATE_PATH = RUN_DIR / "validation_100_gate.json"
if VALIDATION_100_REPORT is None:
    validation_100_pass = False
    validation_100_quality = {"status": "SKIPPED", "pass": False}
    status = "SKIPPED"
else:
    validation_100_quality = validation_quality(VALIDATION_100_REPORT, "validation_100")
    validation_100_pass = bool(validation_100_quality["pass"])
    status = validation_100_quality["status"]
    VALIDATION_100_REPORT["release_quality"] = validation_100_quality
    atomic_json(VALIDATION_100_PATH, VALIDATION_100_REPORT)
    tag_artifact(VALIDATION_100_PATH)

atomic_json(VALIDATION_100_GATE_PATH, {
    "commit_sha": COMMIT_SHA,
    "config_fingerprint": CONFIG_FINGERPRINT,
    **validation_100_quality,
})
tag_artifact(VALIDATION_100_GATE_PATH)
set_gate(
    "validation_100_gate",
    status,
    validation_100_quality,
    {
        "p90_words_max": P90_WORDS_MAX,
        "over_800_rate_max": OVER_800_WORD_RATE_MAX,
        "token_limit_rate_max": TOKEN_LIMIT_RATE_MAX,
        "refusal_rate_max": REFUSAL_RATE_MAX,
        "extractive_fallback_rate_max": EXTRACTIVE_FALLBACK_RATE_MAX,
        "dense_and_reranker_active": True,
    },
    str(VALIDATION_100_GATE_PATH),
)
record_stage("validation_100_gate", status, GATES["validation_100_gate"])
print(json.dumps(validation_100_quality, ensure_ascii=False, indent=2))

## 15. Validation 300

In [ ]:
VALIDATION_300_PATH = RUN_DIR / "validation_300.json"
VALIDATION_300_LOG = RUN_DIR / "validation_300.log"
validation_300_prerequisites = (
    GATES.get("unit_test_gate", {}).get("status") == "PASS"
    and GATES.get("cache_gate", {}).get("status") == "PASS"
    and GATES.get("public_retrieval_gate", {}).get("status") == "PASS"
    and GATES.get("known_problem_retrieval_gate", {}).get("status") == "PASS"
    and GATES.get("smoke_automatic_gate", {}).get("status") == "PASS"
    and GATES.get("validation_100_gate", {}).get("status") == "PASS"
)

if not RUN_VALIDATION_300:
    VALIDATION_300_REPORT = None
    set_gate("validation_300_execution", "SKIPPED", "disabled", "PASS", "RUN_VALIDATION_300=False")
    record_stage("validation_300_execution", "SKIPPED", GATES["validation_300_execution"])
elif not validation_300_prerequisites:
    VALIDATION_300_REPORT = None
    blockers = [
        name for name in (
            "unit_test_gate", "cache_gate", "public_retrieval_gate",
            "known_problem_retrieval_gate", "smoke_automatic_gate", "validation_100_gate",
        )
        if GATES.get(name, {}).get("status") != "PASS"
    ]
    set_gate("validation_300_execution", "SKIPPED", blockers, "Mọi prerequisite PASS", "Validation 300 bị khóa")
    record_stage("validation_300_execution", "SKIPPED", GATES["validation_300_execution"])
    print("Không chạy validation 300; blockers:", blockers)
else:
    if not can_reuse("validation_300_execution", [VALIDATION_300_PATH, VALIDATION_300_LOG]):
        run_stream(
            validation_command("validation_300", VALIDATION_300_PATH),
            cwd=REPO_DIR,
            log_path=VALIDATION_300_LOG,
        )
    VALIDATION_300_REPORT = read_json(VALIDATION_300_PATH)
    VALIDATION_300_REPORT["_release_gate"] = {
        "commit_sha": COMMIT_SHA,
        "config_fingerprint": CONFIG_FINGERPRINT,
    }
    atomic_json(VALIDATION_300_PATH, VALIDATION_300_REPORT)
    tag_artifact(VALIDATION_300_PATH)
    tag_artifact(VALIDATION_300_LOG)
    set_gate("validation_300_execution", "PASS", 300, 300, str(VALIDATION_300_PATH))
    record_stage("validation_300_execution", "PASS", GATES["validation_300_execution"])
    print(f"Validation 300 report: {VALIDATION_300_PATH}")

## 16. Validation 300 và metric comparison gate

In [ ]:
VALIDATION_300_GATE_PATH = RUN_DIR / "validation_300_gate.json"
METRIC_COMPARISON_PATH = RUN_DIR / "metric_comparison.json"

if VALIDATION_300_REPORT is None:
    validation_300_quality = {"status": "SKIPPED", "pass": False}
    validation_300_status = "SKIPPED"
    metric_comparison = {
        "status": "PENDING",
        "pass": False,
        "reason": "Validation 300 chưa chạy.",
    }
else:
    validation_300_quality = validation_quality(VALIDATION_300_REPORT, "validation_300")
    validation_300_status = validation_300_quality["status"]
    VALIDATION_300_REPORT["release_quality"] = validation_300_quality
    atomic_json(VALIDATION_300_PATH, VALIDATION_300_REPORT)
    tag_artifact(VALIDATION_300_PATH)

    new_result = VALIDATION_300_REPORT.get("results", {}).get(MODE, {})
    new_metrics = {
        "competition_meteor": new_result.get("competition_meteor"),
        "competition_rougeL": new_result.get("competition_rougeL"),
    }
    if BASELINE_PATH is None or not BASELINE_PATH.is_file():
        metric_comparison = {
            "status": "PENDING",
            "pass": False,
            "reason": "BASELINE_VALIDATION_300_PATH chưa trỏ tới baseline report tồn tại.",
            "new": new_metrics,
        }
    else:
        baseline = read_json(BASELINE_PATH)
        baseline_config = baseline.get("config", {})
        baseline_result = baseline.get("results", {}).get(MODE, {})
        baseline_metrics = {
            "competition_meteor": baseline_result.get("competition_meteor"),
            "competition_rougeL": baseline_result.get("competition_rougeL"),
        }
        compatible = (
            baseline_config.get("split_name") == "validation_300"
            and baseline_config.get("split_manifest_sha256") == DATASET_HASHES["split_manifest"]
            and all(isinstance(value, (int, float)) for value in baseline_metrics.values())
            and all(isinstance(value, (int, float)) for value in new_metrics.values())
        )
        if not compatible:
            metric_comparison = {
                "status": "PENDING",
                "pass": False,
                "reason": "Baseline không cùng locked split/manifest hoặc thiếu official metrics.",
                "baseline_path": str(BASELINE_PATH),
                "baseline": baseline_metrics,
                "new": new_metrics,
            }
        else:
            metric_pass = (
                float(new_metrics["competition_meteor"]) > float(baseline_metrics["competition_meteor"])
                and float(new_metrics["competition_rougeL"]) > float(baseline_metrics["competition_rougeL"])
            )
            metric_comparison = {
                "status": "PASS" if metric_pass else "FAIL",
                "pass": metric_pass,
                "baseline_path": str(BASELINE_PATH),
                "baseline": baseline_metrics,
                "new": new_metrics,
                "delta": {
                    key: float(new_metrics[key]) - float(baseline_metrics[key])
                    for key in new_metrics
                },
            }

metric_comparison["reference_only_old_floors"] = {
    "meteor_exact_approx": 0.3753115927,
    "rougeL": 0.4274373404,
    "note": "Chỉ tham khảo; không thay official metric gate.",
}
atomic_json(VALIDATION_300_GATE_PATH, {
    "commit_sha": COMMIT_SHA,
    "config_fingerprint": CONFIG_FINGERPRINT,
    **validation_300_quality,
})
atomic_json(METRIC_COMPARISON_PATH, {
    "commit_sha": COMMIT_SHA,
    "config_fingerprint": CONFIG_FINGERPRINT,
    **metric_comparison,
})
tag_artifact(VALIDATION_300_GATE_PATH)
tag_artifact(METRIC_COMPARISON_PATH)

set_gate(
    "validation_300_gate",
    validation_300_status,
    validation_300_quality,
    "Cùng quality gate validation 100",
    str(VALIDATION_300_GATE_PATH),
)
set_gate(
    "metric_improvement_gate",
    metric_comparison["status"],
    metric_comparison,
    "Official competition_meteor và competition_rougeL đều lớn hơn baseline cùng split",
    str(METRIC_COMPARISON_PATH),
)
record_stage("validation_300_gate", validation_300_status, GATES["validation_300_gate"])
record_stage("metric_improvement", metric_comparison["status"], GATES["metric_improvement_gate"])
print(json.dumps(validation_300_quality, ensure_ascii=False, indent=2))
print(json.dumps(metric_comparison, ensure_ascii=False, indent=2))

## 17. Release summary và export

In [ ]:
SUMMARY_PATH = RUN_DIR / "release_gate_summary.json"
REPORT_PATH = RUN_DIR / "release_gate_report.md"
ZIP_PATH = RUN_DIR / "legalqa_release_gate_artifacts.zip"

required_gate_names = [
    "unit_test_gate",
    "cache_gate",
    "public_retrieval_gate",
    "known_problem_retrieval_gate",
    "smoke_automatic_gate",
    "manual_review_gate",
    "validation_100_gate",
    "validation_300_gate",
    "metric_improvement_gate",
]
for gate_name in required_gate_names:
    if gate_name not in GATES:
        set_gate(gate_name, "SKIPPED", "stage did not run", "PASS", "Thiếu stage")

release_ready = all(GATES[name]["status"] == "PASS" for name in required_gate_names)
summary = {
    "commit_sha": COMMIT_SHA,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "dataset_hashes": DATASET_HASHES,
    "gates": {name: GATES[name] for name in required_gate_names},
    "release_ready": release_ready,
    "run_directory": str(RUN_DIR),
}
atomic_json(SUMMARY_PATH, summary)

detail_files = {
    "unit_test_gate": UNIT_LOG,
    "cache_gate": CONFIG_PATH,
    "public_retrieval_gate": PUBLIC_RETRIEVAL_PATH,
    "known_problem_retrieval_gate": RETRIEVAL_GATE_PATH,
    "smoke_automatic_gate": SMOKE_QUALITY_PATH,
    "manual_review_gate": MANUAL_REVIEW_PATH,
    "validation_100_gate": VALIDATION_100_GATE_PATH,
    "validation_300_gate": VALIDATION_300_GATE_PATH,
    "metric_improvement_gate": METRIC_COMPARISON_PATH,
}
rows = [
    "# LegalQA Release Gate",
    "",
    f"- Commit: {COMMIT_SHA}",
    f"- Config fingerprint: {CONFIG_FINGERPRINT}",
    f"- Release ready: {str(release_ready).lower()}",
    "",
    "| Gate | Status | Thực tế | Ngưỡng | File chi tiết |",
    "|---|---|---|---|---|",
]
symbols = {"PASS": "✅ PASS", "FAIL": "❌ FAIL", "PENDING": "🟡 PENDING", "SKIPPED": "⚪ SKIPPED"}
for name in required_gate_names:
    gate = GATES[name]
    actual = json.dumps(gate["actual"], ensure_ascii=False)
    threshold = json.dumps(gate["threshold"], ensure_ascii=False)
    if len(actual) > 180:
        actual = actual[:177] + "..."
    if len(threshold) > 120:
        threshold = threshold[:117] + "..."
    detail = detail_files[name]
    rows.append(
        f"| {name} | {symbols[gate['status']]} | {actual.replace('|', '/')} | "
        f"{threshold.replace('|', '/')} | {detail.name} |"
    )
REPORT_PATH.write_text("\n".join(rows) + "\n", encoding="utf-8")
tag_artifact(SUMMARY_PATH)
tag_artifact(REPORT_PATH)

display(Markdown("\n".join(rows)))

artifact_paths = [
    path for path in RUN_DIR.rglob("*")
    if path.is_file() and path != ZIP_PATH and not path.name.endswith(".tmp")
]
if UNIT_LOG.is_file():
    artifact_paths.append(UNIT_LOG)
if UNIT_META.is_file():
    artifact_paths.append(UNIT_META)
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    seen_names: set[str] = set()
    for path in artifact_paths:
        arcname = path.name if path.parent == RUN_DIR else f"external/{path.name}"
        if arcname in seen_names:
            arcname = f"duplicate-{len(seen_names)}/{path.name}"
        seen_names.add(arcname)
        archive.write(path, arcname=arcname)
tag_artifact(ZIP_PATH)

print(f"Summary: {SUMMARY_PATH}")
print(f"Report: {REPORT_PATH}")
print(f"Bundle: {ZIP_PATH}")
display(FileLink(str(SUMMARY_PATH)))
display(FileLink(str(REPORT_PATH)))
display(FileLink(str(ZIP_PATH)))

if release_ready:
    print("Pipeline đủ điều kiện để chạy full submission trong notebook production.")
else:
    blockers = {
        name: GATES[name]["status"]
        for name in required_gate_names
        if GATES[name]["status"] != "PASS"
    }
    print("Release chưa sẵn sàng:", blockers)
    print("Notebook không chạy full submission.")